# Visualize Multimodal Embeddings

Fetch all vectors from Qdrant, reduce to 2D with t-SNE, and plot clusters colored by modality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from qdrant_client import QdrantClient

from app.config import get_settings

In [ ]:
settings = get_settings()
client = QdrantClient(host=settings.qdrant_host, port=settings.qdrant_port)

# Fetch all points from the collection
result = client.scroll(
    collection_name=settings.collection_name,
    with_vectors=True,
    limit=1000,
)
points = result[0]
print(f"Fetched {len(points)} vectors")

In [ ]:
vectors = np.array([p.vector for p in points])
modalities = [p.payload.get("modality", "unknown") for p in points]
labels = [p.payload.get("source", "") for p in points]

print(f"Vector shape: {vectors.shape}")
print(f"Modalities: {set(modalities)}")

In [ ]:
# Reduce to 2D with t-SNE
perplexity = min(5, len(vectors) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
coords = tsne.fit_transform(vectors)
print(f"t-SNE output shape: {coords.shape}")

In [ ]:
COLORS = {
    "text": "#4CAF50",
    "image": "#2196F3",
    "audio": "#FF9800",
    "video": "#E91E63",
    "pdf": "#9C27B0",
}

fig, ax = plt.subplots(figsize=(10, 8))

for mod in sorted(set(modalities)):
    mask = [m == mod for m in modalities]
    ax.scatter(
        coords[mask, 0],
        coords[mask, 1],
        c=COLORS.get(mod, "gray"),
        label=mod,
        s=120,
        alpha=0.8,
        edgecolors="white",
        linewidth=0.5,
    )

# Add file name labels
for i, label in enumerate(labels):
    ax.annotate(
        label,
        (coords[i, 0], coords[i, 1]),
        fontsize=8,
        alpha=0.7,
        xytext=(5, 5),
        textcoords="offset points",
    )

ax.set_title("Multimodal Embeddings — t-SNE Projection", fontsize=14)
ax.legend(title="Modality", fontsize=10)
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
plt.tight_layout()
plt.show()